In [ ]:
from ultralytics import YOLO
import cv2
import numpy as np
import torch
import os
from tqdm import tqdm
data_collection = 'tim_paper/selected_hands_2'

# Get all video files from inputs folder
def get_video_files(input_dir=f'inputs/{data_collection}', suffix=None):
    video_extensions = ('.mp4', '.MP4', '.avi', '.AVI', '.mov', '.MOV')
    video_files = []
    for file in os.listdir(input_dir):
        if file.endswith(video_extensions):
            video_name = os.path.splitext(file)[0]
            if suffix is None or video_name.endswith(suffix):
                video_files.append({
                    'name': video_name,
                    'path': os.path.join(input_dir, file)
                })
    return video_files

# Input settings
# resolution = (3840, 2160)
show = True
save = True

In [2]:
# Load the YOLO model
model_person = YOLO('checkpoints/yolo/yolo11l.pt')
model = YOLO('checkpoints/yolo/rohan_pretrained.pt')

# Configure model settings
model.conf = 0.5  # NMS confidence threshold
model.iou = 0.5   # NMS IoU threshold

In [3]:
def crop_img(img, box):
    """Crop image based on bounding box"""
    x1, y1, x2, y2 = box
    return img[int(y1):int(y2), int(x1):int(x2)]

def detect_hands_in_frame(frame):
    """Detect hands in a single frame using YOLO model"""
    # Process frame with YOLO
    results = model_person(frame, classes=0)
    # print(results)
    boxes = []
    confidence = []
    
    if len(results) == 0:
        results_hands = model(frame)
        for r in results_hands:
            boxes_tensor = r.boxes.xyxy.cpu()
            confs = r.boxes.conf.cpu()
            for box, conf in zip(boxes_tensor, confs):
                if conf > model.conf:
                    boxes.append(box.flatten())
                    confidence.append(conf)
    # Process detections
    for r in results:
        boxes_tensor = r.boxes.xyxy.cpu()
        confs = r.boxes.conf.cpu()
        for box1, conf in zip(boxes_tensor, confs):
            cropped = crop_img(frame, box1)
            results_hands = model(cropped)
            for r in results_hands:
                boxes_tensor = r.boxes.xyxy.cpu()
                confs = r.boxes.conf.cpu()
                for box2, conf in zip(boxes_tensor, confs):
                    if conf > model.conf:
                        # print(box2)
                        adjusted_box = np.add(np.array(box2).reshape(2, 2), box1[:2])
                        boxes.append(adjusted_box.flatten())
                        confidence.append(conf)
                       
                
    return np.array(boxes) if boxes else np.array([]), np.array(confidence) if confidence else np.array([])

In [ ]:
def process_video(input_file, write_folder=True, show=True):
    """Process video file for hand detection"""
    cap = cv2.VideoCapture(input_file)
    if not cap.isOpened():
        raise FileNotFoundError(f"Cannot open video file: {input_file}")
    
    frame_idx = 0
    output_boxes = []
    out_frame_idx = None
    
    # Setup output directory for frames
    if write_folder:
        parentdir = os.path.dirname(input_file)
        base_name = os.path.splitext(os.path.basename(input_file))[0]
        frames_dir = os.path.join(parentdir, base_name)
        os.makedirs(frames_dir, exist_ok=True)
    
    success = True
    while cap.isOpened() and success:   
        success, frame = cap.read()
        if not success:
            break
            
        # Save frame if requested
        if write_folder:
            output_file = os.path.join(frames_dir, f"{frame_idx:05d}.jpg")
            cv2.imwrite(output_file, frame)
            
        # Detect hands
        boxes, confidence = detect_hands_in_frame(frame)
        
        if len(boxes) > 0:
            output_boxes.append((boxes, confidence, frame_idx))
            
        if show:
            # Visualize detections
            for box, conf in zip(boxes, confidence):
                x1, y1, x2, y2 = map(int, box)
                cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 0, 255), 2)
                cv2.putText(frame, f"Hand, conf: {round(100*conf, 1)}%", (x1 - 30, y1 - 30), cv2.FONT_HERSHEY_PLAIN, 2, (255, 0, 255), 2)
            
            cv2.namedWindow('Detections', cv2.WINDOW_NORMAL)
            cv2.resizeWindow('Detections', 1920, 1080)
            cv2.imshow('Detections', frame)
            if cv2.waitKey(1) == 113:
                success = False

                
        frame_idx += 1
        print(f"Processing frame {frame_idx}", end='\r')
    
    cap.release()
    if show:
        cv2.destroyAllWindows()
        
    return output_boxes

In [ ]:
def save_to_file(boxes, output_file):
    
    # Save output to file
    with open(output_file, 'w+') as f:
        for boxes, conf, frame_idx in boxes:
            f.write(f"{frame_idx} ")
            for i, box in enumerate(boxes):
                f.write(f"{box[0]} {box[1]} {box[2]} {box[3]} {conf[i]} ")
            f.write("\n")


In [ ]:
# Process all videos in the inputs folder
video_files = get_video_files()
print(f"Found {len(video_files)} videos to process")

for video in video_files:
    print(f"\nProcessing video: {video['name']}")
    
    # Process the video
    bboxes = process_video(video['path'], write_folder=False, show=show)
    
    if len(bboxes) == 0:
        print("No hands detected")
    else:
        (boxes, conf, frame_idxs) = zip(*bboxes)
        print(f"\nFound hands at frame {frame_idxs}")
        print(f"Bounding boxes: \n{boxes}")

        if save:
            os.makedirs(f"outputs/{data_collection}", exist_ok=True)
            save_to_file(bboxes, f"outputs/{data_collection}/{video['name']}.txt")

Found 5 videos to process

Processing video: gopro1_selected_hands_2

0: 384x640 (no detections), 81.8ms
Speed: 0.0ms preprocess, 81.8ms inference, 65.0ms postprocess per image at shape (1, 3, 384, 640)
Processing frame 1
0: 384x640 (no detections), 28.0ms
Speed: 3.0ms preprocess, 28.0ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)
Processing frame 2
0: 384x640 (no detections), 51.1ms
Speed: 0.0ms preprocess, 51.1ms inference, 2.1ms postprocess per image at shape (1, 3, 384, 640)
Processing frame 3
0: 384x640 (no detections), 41.9ms
Speed: 2.5ms preprocess, 41.9ms inference, 3.0ms postprocess per image at shape (1, 3, 384, 640)
Processing frame 4
0: 384x640 (no detections), 39.9ms
Speed: 0.0ms preprocess, 39.9ms inference, 2.0ms postprocess per image at shape (1, 3, 384, 640)
Processing frame 5
0: 384x640 (no detections), 28.1ms
Speed: 3.4ms preprocess, 28.1ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)
Processing frame 6
0: 384x640 (no detectio